## 1. Imports

In [1]:
import pandas as pd

import os

## 2. Load Datasets into dataframe

### 2.1 Load of IR-Plag-Dataset

In [2]:
ir_plag_root = "../Dataset/IR-Plag-Dataset"

rows = []

# Reads JAVA file to save in on an attribute of the dataframe
def read_code(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()

# Walks through the dataset to read all the files and save them in a dataframe with their respective label
for current_path, dirs, files in os.walk(ir_plag_root):

    for file in files:
        full_path = os.path.join(current_path, file)

        if "non-plagiarized" in full_path:
            label = 0

        elif "plagiarized" in full_path:
            label = 1

        else:
            continue

        code = read_code(full_path)

        rows.append({
            "code": code,
            "label": label
        })

df_ir_plag = pd.DataFrame(rows)

df_ir_plag.head()

,code,label
0,"/*\n * To change this license header, choose L...",0
1,\n/**\n *\n * @author 65FBEF05E01FAC390CB3FA07...,0
2,\n\n/**\n *\n * @author CB6AB3315634A1E4D11B09...,0
3,"/*\n * To change this license header, choose L...",0
4,public class T01\n{\n public static void mai...,0


### 2.2 Load of conplag_version_2 dataset

In [3]:
conplag_path = "../Dataset/conplag_version_2/versions"

version_folders = [
    "bplag_version_1",
    "bplag_version_2",
    "version_1",
    "version_2"
]

labels = pd.read_csv(conplag_path + "/labels.csv")

train_pairs = pd.read_csv(
    conplag_path + "/train_pairs.csv",
    header=None,
    names=["pair"]
)

test_pairs = pd.read_csv(
    conplag_path + "/test_pairs.csv",
    header=None,
    names=["pair"]
)

# Separate pair into submission 1 and submission 2
# e.g. "00af3420_5449d33c" -> sub1: "00af3420", sub2: "5449d33c"
train_pairs[["sub1", "sub2"]] = train_pairs["pair"].str.split("_", expand=True)
test_pairs[["sub1", "sub2"]] = test_pairs["pair"].str.split("_", expand=True)

# Add labels
# e.g. for pair "00af3420_5449d33c"
#     sub1      sub2        label
# 0   00af3420  5449d33c    1
train_df = train_pairs.merge(labels, on=["sub1", "sub2"], how="left")
test_df = test_pairs.merge(labels, on=["sub1", "sub2"], how="left")

train_df = train_df.rename(columns={"verdict": "label"})
test_df = test_df.rename(columns={"verdict": "label"})

# Read code files for each pair and add them to the dataframe
def read_code(version, pair, sub):
    folder = os.path.join(conplag_path, version, pair, sub)

    if not os.path.exists(folder):
        return None
    
    file_path = os.path.join(folder, sub + ".java")

    if not os.path.exists(file_path):
        return None

    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()
    
conplag_train_dfs = []
conplag_test_dfs = []

for version in version_folders:
    train_temp = train_df.copy()
    test_temp = test_df.copy()

    train_temp["version"] = version
    test_temp["version"] = version

    # Apply the read_code function to each row of the dataframe to create new columns "code1" and "code2" for train and test dataframes
    train_temp["code1"] = train_temp.apply(
            lambda row: read_code(version, row["pair"], row["sub1"]),
            axis=1
        )
    train_temp["code2"] = train_temp.apply(lambda row: read_code(version, row["pair"], row["sub2"]), axis=1)
    test_temp["code1"] = test_temp.apply(lambda row: read_code(version, row["pair"], row["sub1"]), axis=1)
    test_temp["code2"] = test_temp.apply(lambda row: read_code(version, row["pair"], row["sub2"]), axis=1)

    conplag_train_dfs.append(train_temp)
    conplag_test_dfs.append(test_temp)


conplag_train_df = pd.concat(conplag_train_dfs, ignore_index=True)
conplag_test_df = pd.concat(conplag_test_dfs, ignore_index=True)

# keeps only essential columns for training and testing the model
conplag_train_df = conplag_train_df[["code1", "code2", "label"]]
conplag_test_df = conplag_test_df[["code1", "code2", "label"]]

# Data cleaning
conplag_train_df = conplag_train_df.dropna()
conplag_test_df = conplag_test_df.dropna()

conplag_test_df.head()

,code1,code2,label
0,import java.util.*;\n\npublic class Soltion{\n...,import java.util.*;\n\npublic class mentor1 {\...,0
1,import java.io.*;\nimport java.util.*;\n\npubl...,import java.io.*;\nimport java.util.*;\n\npubl...,1
2,import java.util.*;\nimport java.io.*;\npublic...,import java.io.*;\nimport java.util.*;\npublic...,0
3,import java.util.*;\nimport java.io.*;\n\n\npu...,import java.io.*;\nimport java.util.*;\npublic...,0
4,import java.io.*;\nimport java.util.*;\npublic...,import java.io.*;\nimport java.util.*;\n\npubl...,0


In [4]:
print("IR-Plag test dataframe:")
conplag_test_df.head()

IR-Plag test dataframe:


,code1,code2,label
0,import java.util.*;\n\npublic class Soltion{\n...,import java.util.*;\n\npublic class mentor1 {\...,0
1,import java.io.*;\nimport java.util.*;\n\npubl...,import java.io.*;\nimport java.util.*;\n\npubl...,1
2,import java.util.*;\nimport java.io.*;\npublic...,import java.io.*;\nimport java.util.*;\npublic...,0
3,import java.util.*;\nimport java.io.*;\n\n\npu...,import java.io.*;\nimport java.util.*;\npublic...,0
4,import java.io.*;\nimport java.util.*;\npublic...,import java.io.*;\nimport java.util.*;\n\npubl...,0


### 2.3 Load of AI detection dataset

In [5]:
ai_detect_root = "../Dataset/ai_detection"
json_file = os.path.join(ai_detect_root, "java_dataset.jsonl")

# Load dataset
df = pd.read_json(json_file, lines=True)

# Human samples
df_human = pd.DataFrame({
    "code": df["human_code"],
    "label": 0
})

# AI samples (ChatGPT)
df_ai = pd.DataFrame({
    "code": df["chatgpt_code"],
    "label": 1
})

# Join datasets
df_ai_detection = pd.concat(
    [df_human, df_ai],
    ignore_index=True
)

df_ai_detection.head()

,code,label
0,private OptionKindAndValue readKindAndValue() ...,0
1,public <TContinuationResult> Task<TContinuatio...,0
2,public static String getUserAgent() {\n ...,0
3,public String translatePathToLocation( String ...,0
4,"public void process( T image1 , T image2 )\n\t...",0


### 2.4 Load of id2source dataset

In [9]:
id2sourcecode_path = "../Dataset/id2sourcecode"

# Obtain pair of positive cases (plagiarized) and negative cases (non-plagiarized) from the dataset
clone = pd.read_csv(id2sourcecode_path + "/clone_pairs.csv")
nonclone = pd.read_csv(id2sourcecode_path + "/nonclone_pairs.csv")

# Keep only first two columns
clone = clone.iloc[:, 0:2].copy()
nonclone = nonclone.iloc[:, 0:2].copy()

# Rename columns
clone.columns = ["id_1", "id_2"]
nonclone.columns = ["id_1", "id_2"]

# Add labels
clone["label"] = 1
nonclone["label"] = 0

df_pairs = pd.concat(
    [clone, nonclone],
    ignore_index=True
)

print(df_pairs.head())
print(df_pairs["label"].value_counts())

def read_java_file(file_id):
    file_path = os.path.join(
        id2sourcecode_path,
        str(file_id) + ".java"
    )

    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()
    
df_pairs["code_1"] = df_pairs["id_1"].apply(read_java_file)
df_pairs["code_2"] = df_pairs["id_2"].apply(read_java_file)

# keeps only essential columns for training and testing the model
df_plagiarism_detection = df_pairs[["id_1", "id_2", "code_1", "code_2", "label"]].copy()

       id_1      id_2  label
0   8001867  23594635      1
1  15537156  23594635      1
2  20619879  23594635      1
3  16499420  23594635      1
4  20601755  23594635      1
label
0    279032
1    269999
Name: count, dtype: int64


In [10]:
print(df_plagiarism_detection.head())
print(df_plagiarism_detection["label"].value_counts())

       id_1      id_2                                             code_1  \
0   8001867  23594635      private void nioBuild() {\n        try {\n...   
1  15537156  23594635      private void copy(String inputPath, String...   
2  20619879  23594635      public void copyLogic() {\n        if (get...   
3  16499420  23594635      private void saveFile(InputStream in, Stri...   
4  20601755  23594635      public static File copyFile(File file) {\n...   

                                              code_2  label  
0      private void copyFileToPhotoFolder(File ph...      1  
1      private void copyFileToPhotoFolder(File ph...      1  
2      private void copyFileToPhotoFolder(File ph...      1  
3      private void copyFileToPhotoFolder(File ph...      1  
4      private void copyFileToPhotoFolder(File ph...      1  
label
0    279032
1    269999
Name: count, dtype: int64


## 3. Data verification

### 3.1 Verification for IR-Plag-Dataset

In [6]:
print(df_ir_plag["label"].value_counts())

if df_ir_plag.isnull().values.any():
    print("There are null values in the dataset, data cleaning is required.")
else:
    print("There are no null values in the dataset, data is clean.")

label
1    355
0    105
Name: count, dtype: int64
There are no null values in the dataset, data is clean.


### 3.2 Verification for conplag_version_2 dataset

In [7]:
print(conplag_train_df["label"].value_counts())
print(conplag_test_df["label"].value_counts())

if conplag_train_df.isnull().values.any() or conplag_test_df.isnull().values.any():
    print("There are null values in the ConPlag dataset, data cleaning is required.")
else:
    print("There are no null values in the ConPlag dataset, data is clean.")

label
0    330
1    130
Name: count, dtype: int64
label
0    990
1    372
Name: count, dtype: int64
There are no null values in the ConPlag dataset, data is clean.


In [8]:
print(df_ai_detection["label"].value_counts())

if df_ai_detection.isnull().values.any():
    print("There are null values in the dataset, data cleaning is required.")
else:
    print("There are no null values in the dataset, data is clean.")

label
0    221796
1    221796
Name: count, dtype: int64
There are no null values in the dataset, data is clean.


In [11]:
print(df_plagiarism_detection["label"].value_counts())

if df_plagiarism_detection.isnull().values.any():
    print("There are null values in the dataset, data cleaning is required.")
else:
    print("There are no null values in the dataset, data is clean.")

label
0    279032
1    269999
Name: count, dtype: int64
There are no null values in the dataset, data is clean.
